# 03. 구조화된 출력 (Structured Output)

> JSON을 텍스트로 받아 파싱하는 시대는 지났어요. ProviderStrategy vs ToolStrategy, Union 다중 응답, `handle_errors` — V1 구조화 출력 옵션을 정리해요.

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. ProviderStrategy와 ToolStrategy의 차이를 설명하고 적절히 선택할 수 있어요
2. Pydantic, dataclass, TypedDict 세 가지 스키마 방식으로 구조화된 출력을 정의할 수 있어요
3. Union 타입을 사용해 다중 응답 유형을 하나의 에이전트로 처리할 수 있어요
4. handle_errors 옵션으로 검증 오류를 자동 복구하도록 설정할 수 있어요

## 사전 지식

- `02-Tools-V1.ipynb` — V1 Tool 정의와 create_agent 기본 사용법
- Pydantic BaseModel 기초 (타입 힌트, Field)
- Python 타입 힌트 (Literal, Union, TypedDict)

## 구조화된 출력이란?

에이전트가 자연어 대신 **예측 가능한 데이터 구조**로 응답하게 만드는 기능이에요.
JSON 파싱, 정규식 없이도 앱이 바로 사용할 수 있는 객체를 얻을 수 있어요.

> **왜 구조화된 출력이 필요할까요?** 자연어 응답은 사람에게는 좋지만 코드에서 처리하기 어려워요. "평점은 4.5점입니다"라는 텍스트에서 숫자를 뽑아내려면 파싱이 필요하죠. 구조화된 출력은 **자판기**처럼 동작해요 -- 버튼(스키마)을 누르면 정해진 형태(Pydantic 인스턴스)의 결과가 나와요.

```mermaid
flowchart TD
    A["사용자 입력<br/>자연어 텍스트"] --> B["에이전트<br/>create_agent"]
    B --> C{"전략 선택<br/>Strategy"}
    C -->|"네이티브 지원 모델<br/>OpenAI / Anthropic"| D["ProviderStrategy<br/>모델 API 직접 지원"]
    C -->|"범용 모델<br/>Tool Calling 지원"| E["ToolStrategy<br/>도구 호출 기반"]
    C -->|"스키마만 전달<br/>자동 선택"| F["Auto Selection<br/>최적 전략 자동"]
    D --> G["structured_response<br/>반환"]
    E --> G
    F --> G
    G --> H["Pydantic 인스턴스<br/>/ dataclass / dict"]

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef strategy fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e

    class A input
    class B,F process
    class G,H output
    class D,E strategy
```

### 두 가지 전략 비교

| 전략 | 작동 방식 | 지원 모델 | 권장 상황 |
|------|-----------|-----------|----------|
| **ProviderStrategy** | 모델 API가 JSON Schema를 직접 강제 | OpenAI, Anthropic, Grok 등 | 신뢰성 최우선 |
| **ToolStrategy** | 도구 호출(Tool Calling)을 통해 구조화 | Tool Calling 지원 모든 모델 | 범용성 최우선 |
| **Auto (스키마만 전달)** | 모델 능력에 따라 자동 선택 | -- | 대부분의 상황 |

> 🔑 **핵심 개념**: `response_format`에 스키마 클래스를 직접 전달하면 LangChain이 자동으로 최적 전략을 선택해요.  
> 결과는 항상 에이전트 상태의 **`structured_response`** 키로 반환돼요.

### 스키마 정의 방법 비교

| 방법 | 특징 | 반환 타입 | 적합한 상황 |
|------|------|-----------|------------|
| **Pydantic** | 풍부한 검증 (ge/le, Literal, validator) | Pydantic 인스턴스 | 복잡한 검증이 필요할 때 |
| **dataclass** | 경량, 표준 라이브러리, 추가 의존성 없음 | dataclass 인스턴스 | 단순 구조, 가벼운 사용 |
| **TypedDict** | 딕셔너리로 반환, JSON 직렬화 용이 | 딕셔너리 | API 응답, DB 저장 |

> 💡 **실무 팁**: 대부분의 경우 Pydantic을 권장해요. Field의 `description`이 모델이 각 필드를 정확히 채우는 데 결정적인 역할을 해요.

## 환경 설정

In [ ]:
# 환경 변수 로드 (.env 파일에서 OPENAI_API_KEY 등을 읽어요)
from dotenv import load_dotenv

load_dotenv()

In [ ]:
# LangSmith 추적 설정 (선택 사항)
# 구조화된 출력의 재시도 과정을 시각적으로 확인할 수 있어요
import os

# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_PROJECT"] = "LangGraph-V1-05-Structured-Output"

## 1. Provider Strategy — 네이티브 구조화 출력

OpenAI, Anthropic 같은 모델은 API 수준에서 JSON Schema를 직접 강제할 수 있어요.
이 방식이 가장 신뢰성이 높아요 — 모델이 스키마를 **반드시** 따르도록 하드웨어 수준에서 제어해요.

`response_format=스키마클래스`로 전달하면 LangChain이 자동으로 ProviderStrategy를 선택해요.

> 💡 **핵심 정리**: 결과는 `result["messages"]`가 아니라 `result["structured_response"]`에 있어요.  
> 이 점을 강조해서 학생들이 처음부터 올바른 키를 사용하도록 안내하세요.

> ⚠️ **자주 하는 실수**: `result["messages"][-1].content`에서 응답을 찾으려고 해요. 구조화된 출력은 별도의 `structured_response` 키에 있어요.

### 1-1. Pydantic 모델로 연락처 추출

In [2]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

class ContactInfo(BaseModel):
    """연락처 정보를 추출하기 위한 스키마"""
    name: str = Field(description="사람이름")
    email: str = Field(description="이메일")
    phone: str = Field(description="전화번호")

model = init_chat_model("openai:gpt-4o-mini")
agent = create_agent(
    model=model,
    tools=[],
    response_format=ContactInfo
)

# 에이전트 실행
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567",
            }
        ]
    }
)

# 구조화된 응답은 structured_response 키에 있어요
print(type(result["structured_response"]))  # <class 'ContactInfo'> — Pydantic 인스턴스
print(result["structured_response"])


<class '__main__.ContactInfo'>
name='John Doe' email='john@example.com' phone='(555) 123-4567'


In [ ]:
from IPython.display import Image, display

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: START → model → {structured_output} → END
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 1-2. dataclass — 경량 스키마

In [ ]:
from dataclasses import dataclass

@dataclass
class ContactInfoDC:
    """연락처 정보"""
    name: str # 이름 (사람의 이름)
    email: str # 이메일 주소 (사람의 이름)
    phone: str # 전화번호 (사람의 이름)

# dataclass도 response_format에 그대로 전달할 수 있어요
agent_dc = create_agent(
    model=model,
    tools=[],
    response_format=ContactInfoDC,
)

result_dc = agent_dc.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Extract contact info from: Jane Smith, jane@example.com, (555) 987-6543",
            }
        ]
    }
)

print(type(result_dc["structured_response"]))  # <class 'ContactInfoDC'> — dataclass 인스턴스
print(result_dc["structured_response"])


<class '__main__.ContactInfoDC'>
ContactInfoDC(name='Jane Smith', email='jane@example.com', phone='(555) 987-6543')


### 1-3. TypedDict — 딕셔너리 형태 반환

In [ ]:
from typing_extensions import TypedDict

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 2. Tool Strategy — 도구 호출 기반 구조화 출력

ToolStrategy는 도구 호출(Tool Calling)을 통해 구조화된 출력을 얻어요.
**네이티브 지원 여부와 관계없이** Tool Calling을 지원하는 모든 모델에서 동작하므로 범용성이 높아요.

`ToolStrategy`를 명시적으로 import해서 전달하면 강제로 이 전략을 사용해요.

> 🔑 **핵심 개념**: `ToolStrategy`는 내부적으로 스키마를 "도구"로 등록하고, 모델이 그 도구를 호출하게 만들어요.  
> 그래서 Tool Calling을 지원하는 모든 모델에서 동작해요.

> 💡 **핵심 정리**: `Literal` 타입으로 허용값을 제한하고 `ge/le`로 수치 범위를 지정하면, 모델이 스키마를 훨씬 정확히 따라요. 직접 예를 들어서 비교해봅니다.

### 2-1. ToolStrategy로 리뷰 분석

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

# 일반 리뷰 → ProductReview 선택
result_review = agent_union.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Analyze: 'Great product! 5 stars. Fast shipping but a bit expensive.'",
            }
        ]
    }
)
# [리뷰 입력]
print(f"선택된 스키마: {type(result_review['structured_response']).__name__}")
print(result_review["structured_response"])

print()

# 불만 사항 → CustomerComplaint 선택
result_complaint = agent_union.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Customer complaint: Package arrived damaged and contents were broken.",
            }
        ]
    }
)
# [불만 사항 입력]
print(f"선택된 스키마: {type(result_complaint['structured_response']).__name__}")
print(result_complaint["structured_response"])


### 2-2. Union 타입 — 여러 스키마 자동 선택

`Union[SchemaA, SchemaB]`를 사용하면 하나의 에이전트가 여러 종류의 입력을 처리할 수 있어요.
모델이 입력 내용을 보고 **어떤 스키마를 사용할지 자동으로 결정**해요.

> 💡 **실무 팁**: 고객 피드백 처리 시스템처럼 "일반 리뷰"와 "불만 사항"을 구분해서 서로 다른 후속 처리를 해야 할 때 아주 유용해요.

In [ ]:
from typing import Union

# dataclass도 response_format에 그대로 전달할 수 있어요
agent_dc = create_agent(
    model=model,
    tools=[],
    response_format=ContactInfoDC,
)

result_dc = agent_dc.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Extract contact info from: Jane Smith, jane@example.com, (555) 987-6543",
            }
        ]
    }
)

print(type(result_dc["structured_response"]))  # <class 'ContactInfoDC'> — dataclass 인스턴스
print(result_dc["structured_response"])


### 2-3. tool_message_content — 대화 기록 커스터마이징

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 3. 오류 처리 (handle_errors)

모델이 스키마에 맞지 않는 값을 반환할 때 자동으로 수정하는 기능이에요.
예를 들어 평점이 1-5인데 "10/10"으로 입력하면, 자동으로 재시도하여 5로 수정해요.

```mermaid
flowchart LR
    A["모델 출력"] --> B{"스키마 검증"}
    B -->|"통과"| C["structured_response 반환"]
    B -->|"실패<br>ValidationError"| D{"handle_errors 설정"}
    D -->|"True (기본값)"| E["오류 피드백 후 재시도"]
    D -->|"False"| F["예외 발생"]
    D -->|"문자열"| G["커스텀 메시지로 재시도"]
    D -->|"callable"| H["함수가 오류 메시지 생성"]
    E --> A
    G --> A
    H --> A

    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#d4edda,stroke:#28a745,color:#155724
    classDef error fill:#f8d7da,stroke:#dc3545,color:#721c24

    class A,E,G,H process
    class C output
    class F error
```

| handle_errors 값 | 동작 |
|-----------------|------|
| `True` (기본값) | 모든 오류 자동 재시도 |
| `False` | 예외 발생 |
| 문자열 | 해당 메시지로 재시도 |
| 예외 클래스 | 해당 예외만 처리, 나머지는 그대로 발생 |
| callable | 함수가 오류 메시지를 생성해서 재시도 |

> 🔑 **핵심 개념**: `handle_errors=True` (기본값)가 설정되어 있으면 LangSmith에서 재시도 과정을 추적할 수 있어요. 오류 → 피드백 → 재시도 흐름이 그대로 기록돼요.

### 3-1. 자동 재시도 (기본 동작)

In [ ]:
from pydantic import BaseModel, Field

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 3-2. 커스텀 오류 메시지로 재시도

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 기본 오류 메시지보다 더 명확한 안내를 제공해요
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 3-3. 커스텀 오류 핸들러 함수

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 이 셀의 핵심 구현 코드
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 3-4. handle_errors=False — 개발/디버깅 용도

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 4. 종합 실습 — 콘텐츠 추천 시스템

Union 타입과 handle_errors를 결합한 실용적인 예제예요.
책 추천과 영화 추천을 하나의 에이전트로 처리하고, 모델이 입력에 따라 적절한 스키마를 자동 선택해요.

> 💡 **핵심 정리**: 이 패턴이 실무에서 가장 많이 쓰이는 형태예요.  
> 다양한 요청을 받아야 하는 AI 어시스턴트에서 Union 타입이 얼마나 강력한지 직접 실행해서 확인해요.

> 💡 **실무 팁**: `system_prompt`를 통해 에이전트의 역할을 명확히 정의하면 스키마 선택 정확도가 높아져요.

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal, Union

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 5. TODO 실습 — 이력서 파서 만들기

지금까지 배운 내용을 활용해서 이력서에서 정보를 추출하는 에이전트를 만들어 보세요.

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal




## 핵심 요약

이 노트북에서 다음 내용을 배웠어요:

- **ProviderStrategy**: OpenAI, Anthropic 등 네이티브 지원 모델에서 가장 신뢰성 높은 구조화 출력
- **ToolStrategy**: 도구 호출 기반, Tool Calling 지원 모든 모델에서 작동하는 범용 전략
- **Auto Selection**: `response_format=스키마클래스`로 전달하면 최적 전략 자동 선택
- **스키마 방법 3가지**: Pydantic(검증 풍부), dataclass(경량), TypedDict(dict 반환)
- **structured_response 키**: 구조화된 출력은 항상 이 키에서 꺼내야 해요
- **Union 타입**: 여러 스키마를 등록해두면 모델이 입력에 맞는 스키마를 자동 선택
- **handle_errors**: True(기본, 자동 재시도) / False(예외 발생) / 문자열 / callable 설정 가능
- **tool_message_content**: 대화 히스토리에 남길 메시지를 커스터마이징

## 다음 노트북 예고

다음 `04-Streaming-V1.ipynb`에서는 **Stream modes (updates, messages, custom)**를 배워요.  
에이전트의 실행 과정을 실시간으로 스트리밍하고, 노드별로 어떤 메시지가 생성되는지 추적하는 방법을 알아볼게요.